In this lab, we will assemble the nodal finite element discretization of the Poisson problem in 1D. Let's recall the weak formulation:
$$
(\nabla u, \nabla v)_\Omega = (f, v)_\Omega
$$
with $\Omega = [0, 1]$. We impose the following boundary conditions:
$$
u(0) = 0, \qquad \frac{du}{dx}(1) = 0
$$
and consider the source term $f = 1$.

In each of the following blocks, replace the triple dots ... with appropriate Python code.

In [ ]:
import numpy as np

## Gridding
Define the grid with $N = 10$ elements so that $h = 1 / N. Create a numpy array of length $N + 1$ that looks like
$$
x = [0, 0.1, 0.2, ..., 1.0]
$$
Make it dependent on $N$ so that we can change it later. Hint: use either `np.arange` or `np.linspace`.

In [ ]:
N = 10
h = ...
x = ...

## Assembly

We construct the matrix $A$ corresponding to $(\nabla u, \nabla v)_\Omega$ by splitting up the integral over each element.
$$
    (\nabla u, \nabla v)_\Omega = \sum_{i = 0}^N (\nabla u, \nabla v)_{E_i}
$$

Now on a single element, there are only two non-zero basis functions. We therefore need to compute 4 terms, which we can arrange in a $2 \times 2$ matrix. So, for a given element $E = (x_i, x_{i + 1})$, we compute the element matrix
$$
A_{E} = \begin{bmatrix} 
    (\nabla \phi_i, \nabla \phi_i)_{E_i} & (\nabla \phi_{i + 1}, \nabla \phi_i)_{E_i} \\
    (\nabla \phi_{i + 1}, \nabla \phi_i)_{E_i} & (\nabla \phi_{i + 1}, \nabla \phi_{i + 1})_{E_i}
\end{bmatrix}
$$

In [ ]:
A_E = np.array([...])

We are now ready to set up the assembly routine. We will start with a dense implementation in numpy.
- Preallocate A as a zero $N \times N$ matrix using `np.zeros`.
- In a for-loop over all elements:
    - Extract the local degrees of freedom, i.e. the indices of the basis functions that are non-zero on this element.
    - Add the element matrix $A_E$ to the matrix $A$ in the appropriate spots. I.e. for local degrees of freedom $i$ and $j$ make sure that
    $$ A_{ii} \leftarrow A_{ii} + A_{E, 00}$$
    $$ A_{ij} \leftarrow A_{ij} + A_{E, 01}$$
    $$ A_{ji} \leftarrow A_{ji} + A_{E, 10}$$
    $$ A_{jj} \leftarrow A_{jj} + A_{E, 11}$$
    Hint: you can assign submatrices using `np.ix_`.

In [ ]:
A = ...

for i in range(N): 
    loc_dofs = ...
    ...

## The right-hand side
We want to evaluate
$$
    (f, v)_\Omega
$$
for the function $f = 1$. Once again, we split up the integral over the elements and add the contribution from each element to a vector. 

In [ ]:
b = np.zeros(...)

for i in range(...):
    loc_dofs = ...
    ...

## Boundary conditions

Note that we do not have to worry about the natural condition at $x = 1$. However, the essential condition at $x = 0$ requires some work. 
- Create an index array containing the degrees of freedom that are away from the essential boundary condition. These are the `FreeDofs` in ngsolve.
- Extract the submatrix from `A` that corresponds to the `FreeDofs`.
- Extract the subvector from `b` that corresponds to the `FreeDofs`.

In [ ]:
freedofs = ...
A_free = ...
b_free = ...

## Solving

After preallocating `u` as a zeros vector, use the direct linear solver from numpy to solve the system and find the coefficients for the free degrees of freedom. Then set these values in the right indices in the main vector `u`.

In [ ]:
u = np.zeros(...)
u_free = np.linalg.solve(..., ...)

u[...] = ...

## Creating a function

Copy all the code from the previous sections and define a function that takes an input $N$ and outputs the grid and the solution.

In [ ]:
def solve_FEM_Poisson(N):
    ...
    return ...

## Plotting the solution

To plot the solution, we use the plotting tools from `matplotlib` (can be installed via pip install in the terminal).

In [ ]:
import matplotlib.pyplot as plt

u, x = solve_FEM_Poisson(10)

plt.plot(x, u)
plt.show()

Verify that the solution is as expected and respects the boundary conditions. 

We can check out the sparsity pattern of the matrix using the spy function.

In [ ]:
plt.spy(A)
plt.show()

# Part 2. Comparing to the true solution

The true solution is given by
$$
    u_{sol}(x) = \frac12 x (1 - x)
$$

We can interpolate this solution onto the finite element space by specifying the degrees of freedom. Create a vector $\bar{u}_{true}$ given by
$$
    [u_{sol}(x_0), u_{sol}(x_1), u_{sol}(x_2), ... u_{sol}(x_N)]
$$

We can compare our solution to the true solution by computing the $L^2$ norm of the difference. For that, we require the mass matrix that corresponds to
$$
    (u, v)_\Omega
$$

Set up an assembly loop to create the mass matrix `M` using the element matrix
$$
    M_E = \frac{h}{6} \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}
$$

Let `u_h` be the solution we computed. Then we can compute the $L^2$ error as
$$
    \| u_h - u_{sol} \|_\Omega = \sqrt{(u_h - u_{sol}, u_h - u_{sol})_\Omega} = \sqrt{(\bar{u}_h - \bar{u}_{sol}) \cdot (M (\bar{u}_h - \bar{u}_{sol}))}
$$

In [ ]:
difference = u - u_true
error = np.sqrt(...)

Create a function that takes $N$ as an input, computes the solution and returns the error. Make sure to call the previously defined function. 

In [ ]:
def compute_error(u, u_true):
    ...

def solve_and_compute_error(N):
    ...

Let's assume the method convergences with rate $\alpha$, i.e.
$$
    e_h = \| u_h - u_{sol} \|_\Omega \le C h^\alpha
$$

If we then have two solutions $u_1$ and $u_2$ from mesh sizes $h_1$ and $h_2$, then we can compute the convergence rate using the following formula
$$
    \alpha = \frac{\log(e_2 / e_1)}{\log(h_2 / h_1)}
$$

Run the problem for $N = 10$ and $N = 20$ using the defined functions, and compute the convergence rate.

# Part 3. Nonhomogeneous boundary conditions
We now change the boundary conditions such that
$$
    u(0) = 1, \quad \frac{du}{dx}(1) = 1
$$

Derive the weak formulation and create a new function to solve this problem. 

# Part 4: Sparse matrices

The assembled matrix has many zeros, so it is not efficient to save it as a numpy array. As an alternative, we may assemble the matrix as a sparse array. The main difference is that we do not add the submatrices ourselves, but provide the row indices, column indices, and values to the constructor from `scipy.sparse`

In [ ]:
import scipy.sparse as sps

# Element matrix
A_E = ...

# Preallocate the row and column index arrays and the value array
# These should have shape n_elements x 4 so that we can save an element matrix in each row.
rows = np.zeros(...)
cols = np.zeros(...)
vals = np.zeros(...)

# Fill in the row, column, and values using a for loop
for i in range(N):
    rows[i, :] = ...
    cols[i, :] = ...
    vals[i, :] = ...

# Reshape the three arrays to vectors using .flatten()
...

# Use scipy.sparse to do the assembly. This will add values if the indices are the same.
A_sparse = sps.lil_array((vals, (rows, cols)))

Bonus: Vectorize the above code so that there are no longer any for-loops.

We used the `lil_array` (lists of lists) format because we will want to extract a submatrix. Create a new function called `solve_FEM_sparse` that works the same as `solve_FEM_Poisson` but uses the sparse array assembly. The vectors `u` and `f` can remain numpy vectors.

At the solving step, first convert the matrix to `csr` (compressed sparse row) format using `A.tocsr()` and then use `sps.linalg.spsolve` to do the solve.

In [ ]:
...